In [1]:
import joblib

scaler = joblib.load('scaler.joblib')
model = joblib.load('model.joblib')

In [2]:
import ipywidgets as widgets
from IPython.display import display
import pandas as pd

# Assumes 'model' and 'scaler' already loaded
# feature_cols as defined in modelling:
feature_cols = [
    'gender_numeric', 'disability_numeric', 'presentation_feb',
    'education_numeric', 'imd_numeric', 'average_age',
    'num_of_prev_attempts', 'studied_credits', 'avg_score',
    'num_assessments', 'total_clicks'
]

region_cols = [
    'region_East Midlands Region', 'region_Ireland', 'region_London Region',
    'region_North Region', 'region_North Western Region', 'region_Scotland',
    'region_South East Region', 'region_South Region', 'region_South West Region',
    'region_Wales', 'region_West Midlands Region', 'region_Yorkshire Region'
]

feature_cols += region_cols  # complete feature list

education_options = [
    ('No Formal quals', 1),
    ('Lower Than A Level', 2),
    ('A Level or Equivalent', 3),
    ('HE Qualification', 4),
    ('Post Graduate Qualification', 5)
]

imd_options = [
    ('0-10%', 1),
    ('10-20%', 2),
    ('20-30%', 3),
    ('30-40%', 4),
    ('40-50%', 5),
    ('50-60%', 6),
    ('60-70%', 7),
    ('70-80%', 8),
    ('80-90%', 9),
    ('90-100%', 10)
]

region_options = [col.replace('region_', '') for col in region_cols]

presentation_options = [('Oct (J)', 0), ('Feb (B)', 1)]

DEFAULTS = {
    'gender_numeric': 0,  # Female
    'disability_numeric': 0,  # No
    'presentation_feb': 0,  # Oct by default
    'education_numeric': 3,  # A Level or Equivalent
    'imd_numeric': 5,  # 40-50%
    'average_age': 45,
    'num_of_prev_attempts': 0,
    'studied_credits': 60,
    'avg_score': 70.0,
    'num_assessments': 5,
    'total_clicks': 1000,
    'region': 'South East Region'
}

def labeled_widget(label_text, widget, default_checkbox):
    label = widgets.Label(label_text, layout=widgets.Layout(width='160px'))
    widget.layout.width = '250px'
    default_checkbox.layout.margin = '0 20px 0 10px'
    return widgets.HBox([label, widget, default_checkbox])

# Widgets with default checkboxes unchecked (False)
gender_widget = widgets.Dropdown(options=[('Male', 1), ('Female', 0)], value=DEFAULTS['gender_numeric'])
gender_default = widgets.Checkbox(value=False, description='Use default')

disability_widget = widgets.Dropdown(options=[('Yes', 1), ('No', 0)], value=DEFAULTS['disability_numeric'])
disability_default = widgets.Checkbox(value=False, description='Use default')

presentation_widget = widgets.Dropdown(options=presentation_options, value=DEFAULTS['presentation_feb'])
presentation_default = widgets.Checkbox(value=False, description='Use default')

education_widget = widgets.Dropdown(options=education_options, value=DEFAULTS['education_numeric'])
education_default = widgets.Checkbox(value=False, description='Use default')

imd_widget = widgets.Dropdown(options=imd_options, value=DEFAULTS['imd_numeric'])
imd_default = widgets.Checkbox(value=False, description='Use default')

age_widget = widgets.IntSlider(min=18, max=80, step=1, value=DEFAULTS['average_age'])
age_default = widgets.Checkbox(value=False, description='Use default')

prev_attempts_widget = widgets.IntSlider(min=0, max=10, step=1, value=DEFAULTS['num_of_prev_attempts'])
prev_attempts_default = widgets.Checkbox(value=False, description='Use default')

studied_credits_widget = widgets.IntSlider(min=0, max=360, step=30, value=DEFAULTS['studied_credits'])
studied_credits_default = widgets.Checkbox(value=False, description='Use default')

avg_score_widget = widgets.FloatSlider(min=0, max=100, step=0.1, value=DEFAULTS['avg_score'])
avg_score_default = widgets.Checkbox(value=False, description='Use default')

num_assessments_widget = widgets.IntSlider(min=0, max=10, step=1, value=DEFAULTS['num_assessments'])
num_assessments_default = widgets.Checkbox(value=False, description='Use default')

total_clicks_widget = widgets.IntSlider(min=0, max=10000, step=10, value=DEFAULTS['total_clicks'])
total_clicks_default = widgets.Checkbox(value=False, description='Use default')

region_widget = widgets.Dropdown(options=region_options, value=DEFAULTS['region'])
region_default = widgets.Checkbox(value=False, description='Use default')

# Collect widgets
inputs = [
    labeled_widget('Gender:', gender_widget, gender_default),
    labeled_widget('Disability:', disability_widget, disability_default),
    labeled_widget('Presentation:', presentation_widget, presentation_default),
    labeled_widget('Education:', education_widget, education_default),
    labeled_widget('IMD Band:', imd_widget, imd_default),
    labeled_widget('Age:', age_widget, age_default),
    labeled_widget('Previous Attempts:', prev_attempts_widget, prev_attempts_default),
    labeled_widget('Studied Credits:', studied_credits_widget, studied_credits_default),
    labeled_widget('Average Score:', avg_score_widget, avg_score_default),
    labeled_widget('Number of Assessments:', num_assessments_widget, num_assessments_default),
    labeled_widget('Total Clicks:', total_clicks_widget, total_clicks_default),
    labeled_widget('Region:', region_widget, region_default),
]

display(widgets.VBox(inputs))

output = widgets.Output()

def predict_pass_probability(_=None):
    with output:
        output.clear_output()
        data_dict = {
            'gender_numeric': DEFAULTS['gender_numeric'] if gender_default.value else gender_widget.value,
            'disability_numeric': DEFAULTS['disability_numeric'] if disability_default.value else disability_widget.value,
            'presentation_feb': DEFAULTS['presentation_feb'] if presentation_default.value else presentation_widget.value,
            'education_numeric': DEFAULTS['education_numeric'] if education_default.value else education_widget.value,
            'imd_numeric': DEFAULTS['imd_numeric'] if imd_default.value else imd_widget.value,
            'average_age': DEFAULTS['average_age'] if age_default.value else age_widget.value,
            'num_of_prev_attempts': DEFAULTS['num_of_prev_attempts'] if prev_attempts_default.value else prev_attempts_widget.value,
            'studied_credits': DEFAULTS['studied_credits'] if studied_credits_default.value else studied_credits_widget.value,
            'avg_score': DEFAULTS['avg_score'] if avg_score_default.value else avg_score_widget.value,
            'num_assessments': DEFAULTS['num_assessments'] if num_assessments_default.value else num_assessments_widget.value,
            'total_clicks': DEFAULTS['total_clicks'] if total_clicks_default.value else total_clicks_widget.value,
        }

        # Set all region dummy vars to 0 first
        for r in region_cols:
            data_dict[r] = 0
        # Then set chosen region to 1
        chosen_region = DEFAULTS['region'] if region_default.value else region_widget.value
        data_dict[f'region_{chosen_region}'] = 1

        data = pd.DataFrame([data_dict])

        # Reorder columns exactly as in training
        data = data[feature_cols]

        # Scale numeric features
        numeric_features = [
            'education_numeric', 'imd_numeric', 'average_age', 'num_of_prev_attempts',
            'studied_credits', 'avg_score', 'num_assessments', 'total_clicks'
        ]
        data[numeric_features] = scaler.transform(data[numeric_features])

        # Predict
        prob = model.predict_proba(data)[0, 1]
        print(f"Predicted pass probability: {prob:.2%}")

predict_button = widgets.Button(description='Predict Pass Probability')
predict_button.on_click(predict_pass_probability)

display(predict_button, output)


Button(description='Predict Pass Probability', style=ButtonStyle())

Output()

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
df = pd.read_csv("/Users/jamesjackson/Documents/student_outcome_analysis/data/master-table.csv")

In [6]:
df.head()

,id_student,code_module,code_presentation,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,avg_score,num_assessments,total_clicks
0,11391,AAA,2013J,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,82.0,5.0,934.0
1,28400,AAA,2013J,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass,66.4,5.0,1435.0
2,30268,AAA,2013J,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn,NaN,NaN,281.0
3,31604,AAA,2013J,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,N,Pass,76.0,5.0,2158.0
4,32885,AAA,2013J,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,N,Pass,54.4,5.0,1034.0


In [13]:
# --- Filterable Outcome Bar Chart (stable widget wiring) ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1) Styling to match your spec
OUTCOME_ORDER = ['Fail', 'Withdrawn', 'Pass', 'Distinction']
OUTCOME_COLORS = ['#e27d7d', '#9eb3c2', '#a8d5ba', '#2a7f62']  # Fail, Withdrawn, Pass, Distinction

# 2) Ensure binned columns exist (only create if missing)
if 'total_clicks_bin' not in df.columns and 'total_clicks' in df.columns:
    df['total_clicks_bin'] = pd.cut(
        df['total_clicks'],
        bins=[-np.inf, 0, 500, 1000, 2000, 4000, 8000, np.inf],
        labels=['0', '1–500', '501–1000', '1001–2000', '2001–4000', '4001–8000', '8000+']
    )

if 'avg_score_bin' not in df.columns and 'avg_score' in df.columns:
    df['avg_score_bin'] = pd.cut(
        df['avg_score'],
        bins=[-np.inf, 40, 50, 60, 70, 80, 90, 100],
        labels=['<40', '40–50', '50–60', '60–70', '70–80', '80–90', '90–100']
    )

if 'num_assessments_bin' not in df.columns and 'num_assessments' in df.columns:
    df['num_assessments_bin'] = pd.cut(
        df['num_assessments'],
        bins=[-np.inf, 0, 1, 2, 3, 4, 5, 6, 10],
        labels=['0', '1', '2', '3', '4', '5', '6', '7–10']
    )

if 'studied_credits_bin' not in df.columns and 'studied_credits' in df.columns:
    # OU credits are typically discrete (30/60/90/…); treat as categorical labels
    df['studied_credits_bin'] = df['studied_credits'].astype('Int64').astype(str)

if 'num_of_prev_attempts_bin' not in df.columns and 'num_of_prev_attempts' in df.columns:
    df['num_of_prev_attempts_bin'] = pd.cut(
        df['num_of_prev_attempts'],
        bins=[-np.inf, 0, 1, 2, 3, np.inf],
        labels=['0', '1', '2', '3', '4+']
    )

# 3) The 12 filter columns (human-readable)
FILTER_COLUMNS = [
    'region',
    'imd_band',
    'age_band',
    'gender',
    'disability',
    'highest_education',
    'code_presentation',
    'total_clicks_bin',
    'avg_score_bin',
    'num_assessments_bin',
    'studied_credits_bin',
    'num_of_prev_attempts_bin'
]

# Build one dropdown per filter, each with an "All" option
def make_dropdown(col):
    values = df[col].dropna().astype(str).unique().tolist()
    values = sorted(values)
    return widgets.Dropdown(
        options=['All'] + values,
        value='All',
        description=col.replace('_', ' ').title(),
        layout=widgets.Layout(width='280px')
    )

dropdowns = {col: make_dropdown(col) for col in FILTER_COLUMNS}

# Y-axis mode, save controls, and reset
view_mode = widgets.ToggleButtons(
    options=[('Counts', 'count'), ('Percentages', 'pct')],
    value='count',
    description='Y‑axis:',
    layout=widgets.Layout(width='280px')
)

save_toggle = widgets.Checkbox(value=False, description='Save PNG')
save_path = widgets.Text(
    value='/Users/jamesjackson/Documents/student_outcome_analysis/data/Visualisations/outcome_distributions/outcome_bar.png',
    description='Path:',
    layout=widgets.Layout(width='520px')
)

reset_btn = widgets.Button(description='Reset Filters', button_style='')

# Output area for the plot
plot_out = widgets.Output()

# Filtering helper
def apply_filters(dataframe):
    mask = pd.Series(True, index=dataframe.index)
    for col, dd in dropdowns.items():
        val = dd.value
        if val != 'All':
            mask &= dataframe[col].astype(str) == str(val)
    return dataframe[mask]

# Plot function (updates only the output area)
def render_plot(*_):
    with plot_out:
        clear_output(wait=True)

        data = apply_filters(df)

        # Ordered 4-class counts
        counts4 = data['final_result'].value_counts().reindex(OUTCOME_ORDER, fill_value=0).astype(float)

        # Top chart: 4-class (Counts or Percentages, per toggle)
        values_top = counts4.values.copy()
        y_label_top = 'Number of Students'
        title_top = 'Counts of Final Results'
        if view_mode.value == 'pct':
            total_top = values_top.sum()
            values_top = (values_top / total_top * 100.0) if total_top > 0 else values_top
            y_label_top = 'Percentage of Students (%)'
            title_top = 'Percentages of Final Results'

        # Bottom chart: binary percentages (Fail vs Pass)
        fail_count = counts4[['Fail', 'Withdrawn']].sum()
        pass_count = counts4[['Pass', 'Distinction']].sum()
        total_bin = fail_count + pass_count
        if total_bin > 0:
            fail_pct = (fail_count / total_bin) * 100.0
            pass_pct = (pass_count / total_bin) * 100.0
        else:
            fail_pct = pass_pct = 0.0

        # Figure with two rows (top: 4-class, bottom: binary percentage)
        fig, (ax1, ax2) = plt.subplots(
            2, 1, figsize=(8, 8),
            gridspec_kw={'height_ratios': [3, 1]},
            constrained_layout=True
        )

        # ---- Top (4-class) ----
        bars1 = ax1.bar(counts4.index, values_top, color=OUTCOME_COLORS, edgecolor='black')
        ax1.set_title(title_top)
        ax1.set_ylabel(y_label_top)
        ax1.set_xlabel('Final Result')
        ax1.set_xticks(range(len(counts4.index)))
        ax1.set_xticklabels(counts4.index, rotation=0)

        # Labels for top bars
        if view_mode.value == 'count':
            for b, v in zip(bars1, values_top):
                ax1.text(b.get_x() + b.get_width()/2, b.get_height(), f'{int(v):,}', ha='center', va='bottom')
        else:
            for b, v in zip(bars1, values_top):
                ax1.text(b.get_x() + b.get_width()/2, b.get_height(), f'{v:.1f}%', ha='center', va='bottom')

        # ---- Bottom (binary percentage) ----
        bin_labels = ['Fail (Fail + Withdrawn)', 'Pass (Pass + Distinction)']
        bin_values = [fail_pct, pass_pct]
        bin_colors = ['#e27d7d', '#2a7f62']  # red for fail, green for pass

        bars2 = ax2.bar(bin_labels, bin_values, color=bin_colors, edgecolor='black')
        ax2.set_title('Binary Outcome (Percentage)')
        ax2.set_ylabel('Percentage of Students (%)')
        ax2.set_ylim(0, 100)
        ax2.set_xlabel('')

        for b, v in zip(bars2, bin_values):
            ax2.text(b.get_x() + b.get_width()/2, b.get_height(), f'{v:.1f}%', ha='center', va='bottom')

        # Save full figure if requested
        if save_toggle.value:
            fig.savefig(save_path.value, bbox_inches='tight', dpi=150)

        plt.show()


In [15]:
# --- Filterable Outcome Charts (12 filters, stable wiring, stacked horizontal binary bar) ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# 1) Styling to match your spec
OUTCOME_ORDER  = ['Fail', 'Withdrawn', 'Pass', 'Distinction']
OUTCOME_COLORS = ['#e27d7d', '#9eb3c2', '#a8d5ba', '#2a7f62']  # Fail, Withdrawn, Pass, Distinction
FAIL_COLOR = '#e27d7d'
PASS_COLOR = '#2a7f62'

# 2) Ensure binned columns exist (only create if missing)
if 'total_clicks_bin' not in df.columns and 'total_clicks' in df.columns:
    df['total_clicks_bin'] = pd.cut(
        df['total_clicks'],
        bins=[-np.inf, 0, 500, 1000, 2000, 4000, 8000, np.inf],
        labels=['0', '1–500', '501–1000', '1001–2000', '2001–4000', '4001–8000', '8000+']
    )

if 'avg_score_bin' not in df.columns and 'avg_score' in df.columns:
    df['avg_score_bin'] = pd.cut(
        df['avg_score'],
        bins=[-np.inf, 40, 50, 60, 70, 80, 90, 100],
        labels=['<40', '40–50', '50–60', '60–70', '70–80', '80–90', '90–100']
    )

if 'num_assessments_bin' not in df.columns and 'num_assessments' in df.columns:
    df['num_assessments_bin'] = pd.cut(
        df['num_assessments'],
        bins=[-np.inf, 0, 1, 2, 3, 4, 5, 6, 10],
        labels=['0', '1', '2', '3', '4', '5', '6', '7–10']
    )

if 'studied_credits_bin' not in df.columns and 'studied_credits' in df.columns:
    df['studied_credits_bin'] = df['studied_credits'].astype('Int64').astype(str)

if 'num_of_prev_attempts_bin' not in df.columns and 'num_of_prev_attempts' in df.columns:
    df['num_of_prev_attempts_bin'] = pd.cut(
        df['num_of_prev_attempts'],
        bins=[-np.inf, 0, 1, 2, 3, np.inf],
        labels=['0', '1', '2', '3', '4+']
    )

# 3) The 12 filter columns (human-readable)
FILTER_COLUMNS = [
    'region',
    'imd_band',
    'age_band',
    'gender',
    'disability',
    'highest_education',
    'code_presentation',
    'total_clicks_bin',
    'avg_score_bin',
    'num_assessments_bin',
    'studied_credits_bin',
    'num_of_prev_attempts_bin'
]

# Build one dropdown per filter, each with an "All" option
def make_dropdown(col):
    values = df[col].dropna().astype(str).unique().tolist()
    values = sorted(values)
    return widgets.Dropdown(
        options=['All'] + values,
        value='All',
        description=col.replace('_', ' ').title(),
        layout=widgets.Layout(width='280px')
    )

dropdowns = {col: make_dropdown(col) for col in FILTER_COLUMNS}

# Y-axis mode (top chart), save controls, and reset
view_mode = widgets.ToggleButtons(
    options=[('Counts', 'count'), ('Percentages', 'pct')],
    value='count',
    description='Y‑axis:',
    layout=widgets.Layout(width='280px')
)

save_toggle = widgets.Checkbox(value=False, description='Save PNG')
save_path = widgets.Text(
    value='/Users/jamesjackson/Documents/student_outcome_analysis/data/Visualisations/outcome_distributions/outcome_bar.png',
    description='Path:',
    layout=widgets.Layout(width='520px')
)

reset_btn = widgets.Button(description='Reset Filters', button_style='')

# Output area for the plots
plot_out = widgets.Output()

# Filtering helper
def apply_filters(dataframe):
    mask = pd.Series(True, index=dataframe.index)
    for col, dd in dropdowns.items():
        val = dd.value
        if val != 'All':
            mask &= dataframe[col].astype(str) == str(val)
    return dataframe[mask]

# Plot function (updates only the output area)
def render_plot(*_):
    with plot_out:
        clear_output(wait=True)

        data = apply_filters(df)

        # Ordered 4-class counts
        counts4 = data['final_result'].value_counts().reindex(OUTCOME_ORDER, fill_value=0).astype(float)

        # Handle empty selections gracefully
        if counts4.sum() == 0:
            fig, ax = plt.subplots(figsize=(8, 2))
            ax.axis('off')
            ax.text(0.5, 0.5, 'No data for current filters', ha='center', va='center', fontsize=12)
            plt.show()
            return

        # --- Top chart: 4-class (Counts or Percentages) ---
        values_top = counts4.values.copy()
        y_label_top = 'Number of Students'
        title_top = 'Counts of Final Results'
        if view_mode.value == 'pct':
            total_top = values_top.sum()
            values_top = (values_top / total_top * 100.0) if total_top > 0 else values_top
            y_label_top = 'Percentage of Students (%)'
            title_top = 'Percentages of Final Results'

        # --- Bottom chart: binary percentages (Fail vs Pass) ---
        fail_count = counts4[['Fail', 'Withdrawn']].sum()
        pass_count = counts4[['Pass', 'Distinction']].sum()
        total_bin = fail_count + pass_count
        if total_bin > 0:
            fail_pct = (fail_count / total_bin) * 100.0
            pass_pct = (pass_count / total_bin) * 100.0
        else:
            fail_pct = pass_pct = 0.0

        # Figure with two rows (top: 4-class, bottom: 100% horizontal stacked bar)
        fig, (ax1, ax2) = plt.subplots(
            2, 1, figsize=(8, 8),
            gridspec_kw={'height_ratios': [3, 1]},
            constrained_layout=True
        )

        # ---- Top (4-class) ----
        bars1 = ax1.bar(counts4.index, values_top, color=OUTCOME_COLORS, edgecolor='black')
        ax1.set_title(title_top)
        ax1.set_ylabel(y_label_top)
        ax1.set_xlabel('Final Result')
        ax1.set_xticks(range(len(counts4.index)))
        ax1.set_xticklabels(counts4.index, rotation=0)

        # Labels for top bars
        if view_mode.value == 'count':
            for b, v in zip(bars1, values_top):
                ax1.text(b.get_x() + b.get_width()/2, b.get_height(), f'{int(round(v)):,}', ha='center', va='bottom')
        else:
            for b, v in zip(bars1, values_top):
                ax1.text(b.get_x() + b.get_width()/2, b.get_height(), f'{v:.1f}%', ha='center', va='bottom')

        # ---- Bottom (100% horizontal stacked bar) ----
        # Always 0..100 on x; one bar with two stacked segments
        ax2.barh(y=[0], width=[fail_pct], left=[0], color=FAIL_COLOR, edgecolor='black')
        ax2.barh(y=[0], width=[pass_pct], left=[fail_pct], color=PASS_COLOR, edgecolor='black')

        ax2.set_xlim(0, 100)
        ax2.set_ylim(-0.6, 0.6)
        ax2.set_yticks([])
        ax2.set_xlabel('Percentage of Students (%)')
        ax2.set_title('Binary Outcome (Fail vs Pass)')

        # Inline labels inside segments (only if there's room)
        def place_pct_text(ax, left, width, label):
            if width >= 6:  # only label if segment is at least ~6% wide
                ax.text(left + width/2, 0, label, va='center', ha='center', color='white', fontsize=10, fontweight='bold')

        place_pct_text(ax2, 0, fail_pct, f'{fail_pct:.1f}% Fail')
        place_pct_text(ax2, fail_pct, pass_pct, f'{pass_pct:.1f}% Pass')

        # Add thin border around full bar
        ax2.plot([0, 100, 100, 0, 0], [0.3, 0.3, -0.3, -0.3, 0.3], color='black', linewidth=0.8)

        # Save full figure if requested
        if save_toggle.value:
            fig.savefig(save_path.value, bbox_inches='tight', dpi=150)

        plt.show()

# Wire observers (controls displayed once; only the plots update)
for dd in dropdowns.values():
    dd.observe(render_plot, names='value')
view_mode.observe(render_plot, names='value')
save_toggle.observe(render_plot, names='value')
save_path.observe(render_plot, names='value')

def reset_filters(_):
    for dd in dropdowns.values():
        dd.value = 'All'
    view_mode.value = 'count'
    save_toggle.value = False
    render_plot()

reset_btn.on_click(reset_filters)

# Layout and initial render
controls_box = widgets.VBox([
    widgets.HBox([dropdowns['region'], dropdowns['imd_band'], dropdowns['age_band']]),
    widgets.HBox([dropdowns['gender'], dropdowns['disability'], dropdowns['highest_education']]),
    widgets.HBox([dropdowns['code_presentation'], dropdowns['total_clicks_bin'], dropdowns['avg_score_bin']]),
    widgets.HBox([dropdowns['num_assessments_bin'], dropdowns['studied_credits_bin'], dropdowns['num_of_prev_attempts_bin']]),
    widgets.HBox([view_mode, reset_btn]),
    widgets.HBox([save_toggle, save_path])
])

display(controls_box, plot_out)
render_plot()


Output()